# Collections in depth and comprehensions

Collections stop being just storage containers at this level. They become part of how you express an algorithm. Lists, dictionaries, sets, and comprehensions let you shape data in compact ways, but compact code is only good when the underlying idea stays readable.

Comprehensions are especially important because they combine iteration, filtering, and transformation in one expression. Used well, they make intent obvious. Used badly, they hide logic inside a dense one-liner.

The aim of this module is to make you deliberate about both data shape and performance. When you choose a collection or a comprehension style, you are choosing both readability and computational behaviour.

## Visual model

```text
input data -> filter -> transform -> group -> result
```

## How to use this notebook

Read the concept notes first, then run the code cells one at a time. After each run, change an input, prediction, or line of code and rerun it. Intermediate Python becomes easier when you treat every notebook as a place to test a mental model, not just a place to read finished answers.

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.


---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. `dict`: a hash table with compact ordering

Since 3.6, CPython's dict has two parts: a dense array of entries in insertion
order, and a sparse index array of positions into it.

```text
indices : [ _ , 1 , _ , 0 , _ , 2 , _ , _ ]    sparse, sized to load factor
entries : [ (hash, key, value),                dense, INSERTION ORDER
            (hash, key, value),
            (hash, key, value) ]
```


That layout gives ordering for free and saves memory. Insertion order became a
**language guarantee in 3.7** (it was a CPython implementation detail in 3.6 —
worth knowing when reading old code).

| Operation | Complexity |
|---|---|
| `d[k]`, `d[k] = v`, `del d[k]`, `k in d` | O(1) average, O(n) worst |
| iteration | O(n), insertion order |
| `len` | O(1) |

The lookup: hash the key, mask it to an index, probe. On collision, probe again.
On a match of hashes, confirm with `==`. This is why **`__hash__` and `__eq__`
must agree** (Module 09), and why a mutable key would be unfindable (Module 03).

### The methods worth knowing

```text
d.get(k, default)               # no KeyError
d.setdefault(k, [])             # get, inserting the default if absent
d.pop(k, default)
d.popitem()                     # removes and returns the LAST item (LIFO)
d | other                       # merge, 3.9+ (right wins)
d |= other                      # in-place merge
{**a, **b}                      # merge, older syntax
d.keys() / .values() / .items() # VIEWS: live, not copies
dict.fromkeys(seq)              # dedupe preserving order (Module 03)
```


Views are live and cheap:

In [ ]:
keys = d.keys()
d["new"] = 1
print("new" in keys)      # True -- the view reflects the change

Key views also support set operations: `d1.keys() & d2.keys()` gives the common
keys. `d.items() - other.items()` gives the differing pairs. Underused and
excellent.

**`setdefault` versus `defaultdict`:**

In [ ]:
groups = {}
for item in items:
    groups.setdefault(item.kind, []).append(item)     # fine

from collections import defaultdict
groups = defaultdict(list)
for item in items:
    groups[item.kind].append(item)                     # cleaner

The `defaultdict` catch: **reading a missing key inserts it.** `if x in dd`
is safe; `dd[x]` is not. Convert with `dict(dd)` before returning it to code
that does not expect that behaviour.

---

## Concept 5. Comprehensions

In [ ]:
[f(x) for x in xs if pred(x)]           # list
{f(x) for x in xs}                      # set
{k: v for k, v in pairs}                # dict
(f(x) for x in xs)                      # GENERATOR -- lazy, not a tuple

Read them outside-in: *what to produce*, then *what to loop over*, then *what to
keep*.

In [ ]:
[y for x in matrix for y in x]           # flatten: loops in the same order
                                          # you would write them nested
[[y for y in row] for row in matrix]     # nested comprehension: inner produces
                                          # a list per row

The multi-`for` order trips everyone up once. It reads left to right in the same
order as the equivalent nested `for` statements.

### Conditions

In [ ]:
[x for x in xs if x > 0]                 # FILTER: after the for
[x if x > 0 else 0 for x in xs]          # TRANSFORM: a conditional expression
                                          # before the for
[x for x in xs if x > 0 if x < 10]       # two filters, ANDed

### When not to use one

- More than two `for` clauses, or a `for` plus two conditions: use a loop.
- Any side effect. `[print(x) for x in xs]` builds a list of `None` and throws
  it away. Write a `for` loop.
- When the expression no longer fits on a line and reads worse than three lines
  of loop.

A comprehension should read as a *description of the result*. When it starts
reading as a *procedure*, it should be a loop.

### Generator expressions: the lazy version

In [ ]:
sum(x**2 for x in range(1_000_000))     # never builds the list
any(line.startswith("ERROR") for line in fh)   # stops at the first hit
max((score(x), x) for x in candidates)

Parentheses are optional when it is the only argument to a call. Use a generator
expression when you are consuming the values once — it uses O(1) memory instead
of O(n) and can short-circuit. Module 14 makes this a design tool.

---

## Concept 6. Sorting

In [ ]:
sorted(xs)                                   # new list
xs.sort()                                    # in place, returns None
sorted(xs, key=len)                          # by a computed value
sorted(xs, key=lambda p: (p.dept, -p.score)) # multi-key; - reverses a number
sorted(xs, reverse=True)
sorted(xs, key=str.casefold)                 # case-insensitive text

from operator import attrgetter, itemgetter
sorted(people, key=attrgetter("age"))        # faster and clearer than a lambda
sorted(rows, key=itemgetter(1, 0))

Facts to keep:

- **Timsort, O(n log n), and stable.** Stability means equal elements keep their
  relative order, which is what makes multi-pass sorting work:

  ```python
  rows.sort(key=itemgetter("name"))     # secondary key first
  rows.sort(key=itemgetter("dept"))     # primary key last
  ```

- The `key` function is called **once per element**, not on every comparison.
  That is why `key=` beats the removed `cmp=` and why an expensive key is fine.
- For reverse-sorting on a non-numeric key, use `reverse=True` rather than
  negating — you cannot negate a string.
- For top-k, `heapq.nlargest(k, xs)` is O(n log k) and streams its input.

---

## Concept 8. Choosing a container

| Need | Use |
|---|---|
| Ordered, changes | `list` |
| Fixed record, hashable | `tuple` / `NamedTuple` / frozen dataclass |
| Lookup by key | `dict` |
| Membership, dedupe, set algebra | `set` |
| Queue, sliding window, last-N | `deque` |
| Counting | `Counter` |
| Grouping | `defaultdict(list)` |
| Priority / top-k | `heapq` |
| Layered config | `ChainMap` |
| Sorted, with fast inserts | `bisect` on a list, or `sortedcontainers` |
| Large numeric data | `array`, or NumPy (Module 29) |

The two questions that answer this almost every time:

1. **How will I look things up?** By position → list. By key → dict. By presence
   → set.
2. **Where do I add and remove?** Both ends → deque. End only → list.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: `list`: a dynamic array of pointers
- Section 2: `dict`: a hash table with compact ordering
- Section 3: `set`: a hash table without values
- Section 4: `tuple`
- Section 5: Comprehensions
- Section 6: Sorting
- Section 7: `collections`
- Section 8: Choosing a container

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import random
from collections.abc import Iterable
from dataclasses import dataclass
from datetime import date, timedelta

---

## `Request`

_Request_

In [ ]:
@dataclass(frozen=True)
class Request:
    day: date
    path: str
    user: str
    status: int
    ms: int
    bytes_out: int

---

## `generate`

_generate_

In [ ]:
def generate(n: int = 5000, seed: int = 7) -> list[Request]:
    rng = random.Random(seed)
    start = date(2026, 7, 27)
    paths = ["/", "/login", "/api/items", "/api/items/1", "/search",
             "/static/app.js", "/admin"]
    users = [f"u{i}" for i in range(120)] + ["anon"] * 40
    out = []
    for _ in range(n):
        path = rng.choices(paths, weights=[30, 10, 25, 15, 12, 20, 2])[0]
        status = rng.choices([200, 200, 200, 301, 404, 500],
                             weights=[70, 10, 5, 5, 8, 2])[0]
        out.append(Request(
            day=start + timedelta(days=rng.randrange(7)),
            path=path,
            user=rng.choice(users),
            status=status,
            ms=max(1, int(rng.lognormvariate(3.2, 0.9))),
            bytes_out=rng.randrange(200, 50_000),
        ))
    return out

---

## `status_counts`

How many requests per status code, most common first.

In [ ]:
def status_counts(requests: Iterable[Request]) -> dict[int, int]:
    """How many requests per status code, most common first.
    Container: ?"""
    raise NotImplementedError

---

## `requests_by_day`

Group requests by day. Container: ?

In [ ]:
def requests_by_day(requests: Iterable[Request]) -> dict[date, list[Request]]:
    """Group requests by day. Container: ?"""
    raise NotImplementedError

---

## `top_paths`

The k most requested paths with their counts.

In [ ]:
def top_paths(requests: Iterable[Request], k: int = 3) -> list[tuple[str, int]]:
    """The k most requested paths with their counts.
    There is a one-method answer. Find it."""
    raise NotImplementedError

---

## `slowest_per_path`

The worst latency seen for each path.

In [ ]:
def slowest_per_path(requests: Iterable[Request]) -> dict[str, int]:
    """The worst latency seen for each path.
    Do it in ONE pass. A dict of running maxima, not a group-then-max."""
    raise NotImplementedError

---

## `percentiles`

Compute percentile latencies.

In [ ]:
def percentiles(values: list[int], ps: tuple[float, ...] = (50, 95, 99)) -> dict[float, int]:
    """Compute percentile latencies.

    Sort once, index. Then answer in a comment:
      - why is p95 a more useful SLO than the mean?
      - why does averaging the p95 of each server give the wrong overall p95?
    (Both questions come back in Module 35.)
    """
    raise NotImplementedError

---

## `users_seeing_errors`

The set of users who received any 5xx response.

In [ ]:
def users_seeing_errors(requests: Iterable[Request]) -> set[str]:
    """The set of users who received any 5xx response."""
    raise NotImplementedError

---

## `churned_users`

Users active on first_day but NOT on last_day.

In [ ]:
def churned_users(requests: Iterable[Request], first_day: date,
                  last_day: date) -> set[str]:
    """Users active on first_day but NOT on last_day.

    This is set difference. Write it as one expression using set algebra, and
    note in a comment what the loop-based version would have cost."""
    raise NotImplementedError

---

## `daily_report`

Render an aligned table: one row per day, with columns for request

In [ ]:
def daily_report(requests: Iterable[Request]) -> str:
    """Render an aligned table: one row per day, with columns for request
    count, error rate as a percentage, p95 latency, and total megabytes out.

    Sort by day. Use the format mini-language for alignment (Module 03), not
    manual padding. Requests may arrive as an ITERATOR, so consume it once.
    """
    raise NotImplementedError

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    reqs = generate()

    counts = status_counts(reqs)
    assert sum(counts.values()) == len(reqs)
    assert list(counts)[0] == 200, "most common status should come first"

    by_day = requests_by_day(reqs)
    assert len(by_day) == 7
    assert sum(len(v) for v in by_day.values()) == len(reqs)

    top = top_paths(reqs, 3)
    assert len(top) == 3 and top[0][1] >= top[1][1] >= top[2][1]

    slowest = slowest_per_path(reqs)
    assert set(slowest) <= {r.path for r in reqs}

    p = percentiles([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
    assert p[50] <= p[95] <= p[99]

    errs = users_seeing_errors(reqs)
    assert all(isinstance(u, str) for u in errs)

    churned = churned_users(reqs, date(2026, 7, 27), date(2026, 8, 2))
    assert isinstance(churned, set)

    report = daily_report(reqs)
    assert report.count("\n") >= 7, report

    print("all checks passed\n")
    print(report)

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.